# report_labels 产物分析（含 Actual 版）

本 notebook **只读分析** `artifacts/report_labels/` 下本次运行生成的 5 个产物：

| 文件 | 内容 |
| --- | --- |
| `reports.parquet` | 清洗后的报告表（含正文 text） |
| `report_fy_labels.parquet` | 报告 × 预测财年行（Residual / Edge / 发布前共识） |
| `report_confirmation_labels.parquet` | 未来 1/3 月 confirmation（fixed/market/active 面板） |
| `label_coverage_audit.csv` | 覆盖率与失效原因长表 |
| `label_metadata.json` | 构建配置、counts、运行时长 |

相比仓库里旧的 `report_labels_analysis.ipynb`，本版对应**已接入 Actual 数据**的最新产物。



In [ ]:
%matplotlib inline
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ---- matplotlib 中文显示 ----
for _fp in [
    "/usr/share/fonts/wqy-microhei/wqy-microhei.ttc",
    "/usr/share/fonts/wqy-zenhei/wqy-zenhei.ttc",
    "/usr/share/fonts/cjkuni-uming/uming.ttc",
]:
    if Path(_fp).exists():
        font_manager.fontManager.addfont(_fp)
plt.rcParams["font.sans-serif"] = [
    "WenQuanYi Micro Hei", "WenQuanYi Zen Hei", "AR PL UMing CN", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

# 自动定位仓库根：从当前目录向上查找，并兜底常见绝对路径。
def _locate_root():
    marker = Path("artifacts") / "report_labels" / "label_metadata.json"
    candidates = []
    p = Path.cwd().resolve()
    for _ in range(6):
        candidates.append(p)
        p = p.parent
    candidates += [
        Path.home() / "Projects" / "chinese-wwm-roberta",
        Path("/home/intern_fjq_2026/Projects/chinese-wwm-roberta"),
    ]
    for c in candidates:
        if (c / marker).exists():
            return c
    return None

ROOT = _locate_root()
if ROOT is None:
    raise SystemExit("找不到仓库根目录，请手动设置 ROOT = Path('<仓库绝对路径>')")
ART = ROOT / "artifacts" / "report_labels"
print("产物目录:", ART.resolve())



In [ ]:
# 1) 构建元信息
meta = json.loads((ART / "label_metadata.json").read_text(encoding="utf-8"))
cfg = meta["configuration"]
print("label_version :", meta["label_version"], "| created_at:", meta["created_at"])
print("label 窗口     :", cfg["label_start"], "->", cfg["label_end"])
print("读取窗口      :", cfg["read_start"], "->", cfg["read_end"])
print("calendar      :", cfg["calendar_date_column"], cfg["calendar_start"], "->", cfg["calendar_end"])
print()
print("=== counts ===")
for k, v in meta["counts"].items():
    print(f"  {k:32s} {v:,}")
print()
print("actual_scale_median_ratio:", meta.get("actual_scale_median_ratio"))



In [ ]:
# 2) 覆盖率审计长表
audit = pd.read_csv(ART / "label_coverage_audit.csv")
print("audit 长表:", audit.shape, "| 字段:", list(audit.columns))
print()

# 顶层:label x status 的汇总(跨年份/前瞻期/面板求和)
top = audit.groupby(["label", "status"])["count"].sum().unstack(fill_value=0)
print(top.astype(int))



In [ ]:
# 3) 读取三张 parquet 与 schema 概览
rep = pd.read_parquet(ART / "reports.parquet")
fy = pd.read_parquet(ART / "report_fy_labels.parquet")
cf = pd.read_parquet(ART / "report_confirmation_labels.parquet")

for name, df in [("reports", rep), ("report_fy_labels", fy), ("report_confirmation_labels", cf)]:
    print(f"==== {name}: {df.shape} ====")
    print("  columns:", ", ".join(df.columns))
    print()



## 一、`reports.parquet` —— 报告文本与元信息


In [ ]:
print("唯一值: report_id =", rep["report_id"].nunique(),
      "| stock_code =", rep["stock_code"].nunique(),
      "| org_id =", rep["org_id"].nunique())
print("唯一 report_id < 行数?" , rep["report_id"].nunique() < len(rep))
print()
print("发布日期 -> 可用交易日 延迟分布(天):")
print((rep["available_date"] - rep["publish_date"]).dt.days.value_counts().sort_index().head(10))
print()
print("text 长度(字符) 分位数:")
print(rep["text"].str.len().describe([0.5, 0.9, 0.99]).round(0).astype(int).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].hist(rep["text"].str.len().clip(0, 30000), bins=60, color="steelblue")
axes[0].set_title("text 字符长度分布(截断 3w)")
monthly = rep["available_date"].dt.to_period("M").value_counts().sort_index()
axes[1].bar([str(p) for p in monthly.index], monthly.values, color="tomato")
axes[1].set_title("按可用交易月报告数")
axes[1].tick_params(axis="x", rotation=90)
org = rep["org_id"].value_counts().head(10)
axes[2].barh(org.index[::-1], org.values[::-1], color="seagreen")
axes[2].set_title("机构 Top10(报告数)")
plt.tight_layout(); plt.show()



## 二、`report_fy_labels.parquet` —— 报告 × 预测财年


In [ ]:
print("report_fy_labels:", fy.shape, "| 唯一 report_id:", fy["report_id"].nunique())
print()
print("前瞻期 forecast_horizon 分布:")
print(fy["forecast_horizon"].value_counts().sort_index().to_string())
print()
print("每篇报告覆盖的 FY 数(1~3) 分布:")
print(fy.groupby("report_id").size().value_counts().sort_index().to_string())



In [ ]:
# 发布前共识: 状态 / 有效位 / 无效原因
print("pre_state(发布前共识盈利状态):")
print(fy["pre_state"].value_counts(dropna=False).to_string())
print()
print("pre_label_valid =", int(fy["pre_label_valid"].sum()), "/", len(fy),
      f"({fy['pre_label_valid'].mean():.3%})")
print()
print("pre_label_invalid_reason:")
print(fy["pre_label_invalid_reason"].value_counts(dropna=False).to_string())
print()
print("n_org_pre(发布前同行数) 分位数:", fy["n_org_pre"].quantile([0.5, 0.9, 0.99]).round(2).tolist())



In [ ]:
# Actual 接入 + Residual/Edge 有效位
print("actual_np 覆盖: 非空", int(fy["actual_np"].notna().sum()), "/", len(fy),
      f"({fy['actual_np'].notna().mean():.3%})")
print("residual_valid =", int(fy["residual_valid"].sum()), f"({fy['residual_valid'].mean():.3%})")
print("edge_valid     =", int(fy["edge_valid"].sum()), f"({fy['edge_valid'].mean():.3%})")
print()
print("actual_invalid_reason:")
print(fy["actual_invalid_reason"].value_counts(dropna=False).to_string())
print()
print("actual_transition_state(发布前共识 -> 实际):")
print(fy["actual_transition_state"].value_counts(dropna=False).to_string())



In [ ]:
# Residual / Edge / 误差 分布
v = fy[fy["residual_valid"] == 1]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].hist(v["residual_signed_raw"].clip(-1, 1), bins=80, color="steelblue")
axes[0].set_title("residual_signed_raw\n(actual-forecast)/scale, 截断[-1,1]")
axes[1].hist(v["report_abs_error"].clip(0, 1), bins=80, color="tomato")
axes[1].set_title("report_abs_error\n|actual-forecast|/scale")
axes[2].hist(v["edge_raw"].clip(-1, 1), bins=80, color="seagreen")
axes[2].set_title("edge_raw = 共识误差 - 报告误差")
axes[3].bar(*zip(*v["edge_sign"].value_counts().items()))
axes[3].set_title("edge_sign(报告是否跑赢共识)")
plt.tight_layout(); plt.show()

print("residual_signed_raw 均值:", round(v["residual_signed_raw"].mean(), 4))
print("report_abs_error 均值 :", round(v["report_abs_error"].mean(), 4))
print("consensus_abs_error 均值:", round(v["consensus_abs_error"].mean(), 4))
print("edge>0(报告更准) 占比:", round((v["edge_raw"] > 0).mean(), 4))



In [ ]:
# 报告 vs 共识 绝对误差对比(点云 + 对角线)
v = fy[fy["residual_valid"] == 1]
re = v["report_abs_error"]; ce = v["consensus_abs_error"]
plt.figure(figsize=(6, 6))
plt.hexbin(ce.clip(0, 3), re.clip(0, 3), gridsize=60, cmap="Blues", mincnt=1)
plt.plot([0, 3], [0, 3], "r--", lw=1, label="y=x(两者相等)")
plt.xlabel("consensus_abs_error(次数/scale, 截断3)"); plt.ylabel("report_abs_error(截断3)")
plt.title("报告 vs 同行共识 的预测误差")
plt.legend(); plt.colorbar(); plt.show()

print("相关性 corr(report_abs_error, consensus_abs_error):", round(np.corrcoef(re, ce)[0, 1], 4))
print("days_to_actual 中位数(天):", int(v["days_to_actual"].median()))



In [ ]:
# 尺度与离散度体检
print("forecast_new(元) 范围:", float(fy["forecast_new"].quantile(0.01)), "->", float(fy["forecast_new"].quantile(0.99)))
print("scale_floor 分位数:", fy["scale_floor"].quantile([0.5, 0.9, 0.99]).round(0).tolist())
print("scale_t(=max(|consensus_pre|, floor)) 分布:")
print(fy["scale_t"].describe([0.5, 0.9, 0.99]).round(0).to_string())
print()
print("dispersion_pre(归一化 MAD) 分位数:", fy["dispersion_pre"].quantile([0.5, 0.9, 0.99]).round(3).tolist())
print("scale_reference 计数:")
print(fy["scale_reference"].value_counts(dropna=False).to_string())



## 三、`report_confirmation_labels.parquet` —— 未来共识收敛


In [ ]:
print("confirmation:", cf.shape, "| 唯一 report_id:", cf["report_id"].nunique())
print()
# 有效位: 面板 x 确认月
piv_valid = cf.pivot_table(index="confirmation_months", columns="peer_panel",
                           values="confirmation_valid", aggfunc="mean") * 100
print("confirmation_valid 率(%) 面板 x 月份:")
print(piv_valid.round(2).to_string())
print()
piv_probe = cf.pivot_table(index="confirmation_months", columns="peer_panel",
                           values="confirmation_probe_valid", aggfunc="mean") * 100
print("confirmation_probe_valid 率(%):")
print(piv_probe.round(2).to_string())
print()
print("invalid_reason Top 分布:")
print(cf["invalid_reason"].value_counts(dropna=False).head(12).to_string())



In [ ]:
# progress(未来共识朝报告方向收敛的程度) 与 离散度变化
cv = cf[cf["confirmation_valid"] == 1]
print("progress_raw 均值(全 valid):", round(cv["progress_raw"].mean(), 4))
print("progress_clipped 分位数:", cv["progress_clipped"].quantile([0.1, 0.5, 0.9]).round(3).tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cv["progress_clipped"].clip(-1, 2), bins=80, color="steelblue")
axes[0].set_title("progress_clipped(截断到clip)")
axes[1].hist(cv["delta_log_dispersion"].clip(-2, 2), bins=80, color="tomato")
axes[1].set_title("delta_log_dispersion\nlog(未来分散度/发布前)")
plt.tight_layout(); plt.show()

# 按面板画 progress_clipped
fig2, axes2 = plt.subplots(1, 3, figsize=(17, 4))
for ax, panel in zip(axes2, ["fixed", "market", "active"]):
    d = cv[cv["peer_panel"] == panel]["progress_clipped"].clip(-1, 2)
    ax.hist(d, bins=60, color="seagreen")
    ax.set_title(f"{panel}: progress_clipped")
    ax.axvline(0, color="r", lw=1)
plt.tight_layout(); plt.show()



In [ ]:
# 同行规模 / 样本权重体检
print("n_org_pre 分位数:", cf["n_org_pre"].quantile([0.5, 0.9, 0.99]).round(0).tolist())
print("n_org_future 分位数:", cf["n_org_future"].quantile([0.5, 0.9, 0.99]).round(0).tolist())
print("n_peer_updates(有更新的机构数) 分布:")
print(cf["n_peer_updates"].value_counts().head(8).sort_index().to_string())
print()
cv = cf[cf["confirmation_valid"] == 1]
print("sample_weight 分位数:", cv["sample_weight"].quantile([0.5, 0.9, 0.99]).round(3).tolist())



## 四、关键结论（本次运行，2024 年窗口）


In [ ]:
print("=== 关键数字回顾 ===")
print(f"报告数(reports_output)      : {meta['counts']['reports_output']:,}")
print(f"报告×FY 行(report_fy)       : {meta['counts']['report_fy_rows_output']:,}")
print(f"Residual/Edge 有效          : {meta['counts']['residual_valid']:,}")
print(f"confirmation 有效           : {meta['counts']['confirmation_valid']:,}")
print(f"confirmation probe 有效     : {meta['counts']['confirmation_probe_valid']:,}")
print(f"actual_scale_median_ratio   : {meta['actual_scale_median_ratio']:.3f}  (在[0.2,5]内=单位对齐)")
edge_win = (fy.loc[fy['edge_valid'] == 1, 'edge_raw'] > 0).mean()
print(f"edge>0 占比                 : {edge_win:.3f}")



> 解读口径（便于写报告）：
> - `residual_signed_raw = (actual - forecast) / scale_t`：>0 表示报告**低估**了净利润。
> - `edge_raw = 共识误差绝对值 - 报告误差绝对值`：>0 表示该报告比同行共识**更接近真实值**。
> - `progress_clipped`：未来 1/3 月同行共识朝目标报告方向收敛的程度；<0 表示市场后续**反着**走。
> - `delta_log_dispersion < 0` 表示发布后同行分歧**收窄**（趋同）；>0 表示分歧放大。

